In [1]:
import requests
import json
import os
from datetime import datetime, timezone
import random
import asyncio, aiohttp
import aiolimiter
from tqdm.auto import tqdm
from tqdm.asyncio import tqdm_asyncio


/Users/jonasbjaerke/Projects/data_analysis_practice/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def load_posts_dict(file_paths):
    """
    Takes a list of .jsonl file paths and returns a dict:

        {
            <source_name>: { post_uri -> post_data },
            ...
        }

    where source_name is derived from the filename (without extension).
    """

    all_posts = {}

    for file_path in file_paths:
        source_name = os.path.splitext(os.path.basename(file_path))[0]

        posts_dict = {}
        bad_lines = 0

        with open(file_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    post = json.loads(line)
                except json.JSONDecodeError:
                    bad_lines += 1
                    continue

                post_id = (
                    post.get("uri")
                    or post.get("post", {}).get("uri")
                    or post.get("id")
                )

                if not post_id:
                    continue

                # optional: annotate post with source
                post["_source"] = source_name

                posts_dict[post_id] = post

        all_posts[source_name] = posts_dict

        print(
            f"Loaded {len(posts_dict)} posts from {file_path} "
            f"({bad_lines} bad lines skipped)"
        )

    return all_posts


In [3]:
files = [
    "aew.jsonl",
    "AI.jsonl",
    "booksky.jsonl",
    "gaza.jsonl",
    "GoldenGlobes.jsonl",
    "Minneapolis.jsonl",
    "tennis.jsonl",
    "Trump.jsonl",
    "TheTraitors.jsonl",
    "releasetheepsteinfiles.jsonl",
]


posts = load_posts_dict(files)


Loaded 20000 posts from aew.jsonl (0 bad lines skipped)
Loaded 20000 posts from AI.jsonl (0 bad lines skipped)
Loaded 19830 posts from booksky.jsonl (0 bad lines skipped)
Loaded 20000 posts from gaza.jsonl (0 bad lines skipped)
Loaded 8411 posts from GoldenGlobes.jsonl (0 bad lines skipped)
Loaded 20000 posts from Minneapolis.jsonl (0 bad lines skipped)
Loaded 7651 posts from tennis.jsonl (0 bad lines skipped)
Loaded 20000 posts from Trump.jsonl (0 bad lines skipped)
Loaded 20000 posts from TheTraitors.jsonl (0 bad lines skipped)
Loaded 16588 posts from releasetheepsteinfiles.jsonl (0 bad lines skipped)


In [78]:
import asyncio
import aiohttp
from datetime import datetime, timezone
from tqdm import tqdm
import aiolimiter


REPOST_API  = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"
PROFILE_API = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
FEED_API    = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
FOLLOW_API  = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

HEADERS = {"User-Agent": "repost-prediction-research/1.0"}


# --------------------------------------------------
# UTILS
# --------------------------------------------------

def parse_dt(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)

def get_author_dids(posts_dict):
    out = set()
    for p in posts_dict.values():
        if not p:
            continue
        did = (
            p.get("author", {}).get("did")
            or p.get("post", {}).get("author", {}).get("did")
        )
        if did:
            out.add(did)
    return out

def flatten_posts_dict(posts_by_tag):
    flat_posts = {}
    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            flat_posts[uri] = post
    return flat_posts


# --------------------------------------------------
# FETCH REPOSTERS (ROBUST)
# --------------------------------------------------

async def fetch_reposters_safe(session, uri, retries=3):
    for attempt in range(retries):
        try:
            async with session.get(
                REPOST_API,
                params={"uri": uri, "limit": 100},
                headers=HEADERS
            ) as r:
                if r.status == 200:
                    data = await r.json()
                    return [
                        u["did"]
                        for u in data.get("repostedBy", [])
                        if "did" in u
                    ]
        except (aiohttp.ClientError, asyncio.TimeoutError):
            pass

        # exponential backoff
        await asyncio.sleep(0.3 * (2 ** attempt))

    return []


async def collect_all_reposters(posts_dict, concurrency=50):
    connector = aiohttp.TCPConnector(
        limit=100,
        limit_per_host=concurrency,
        ttl_dns_cache=300
    )

    timeout = aiohttp.ClientTimeout(total=60, connect=10, sock_read=30)
    sem = asyncio.Semaphore(concurrency)

    reposters = set()
    no_reposter_count = 0

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout
    ) as session:

        async def fetch_with_uri(uri):
            async with sem:
                reposted_by = await fetch_reposters_safe(session, uri)
                return uri, reposted_by

        uris = [
            uri for uri, p in posts_dict.items()
            if p and p.get("repostCount", 0) > 0
        ]

        CHUNK_SIZE = 500
        total = len(uris)

        with tqdm(total=total, desc="Reposters", unit="post") as pbar:
            for i in range(0, total, CHUNK_SIZE):
                chunk = uris[i:i + CHUNK_SIZE]
                tasks = [fetch_with_uri(uri) for uri in chunk]

                results = await asyncio.gather(
                    *tasks, return_exceptions=True
                )

                for result in results:
                    if isinstance(result, Exception):
                        pbar.update(1)
                        continue

                    uri, reposted_by = result
                    posts_dict[uri]["reposted_by"] = reposted_by

                    if not reposted_by:
                        posts_dict[uri]["repost_user_unobservable"] = True
                        no_reposter_count += 1
                    else:
                        posts_dict[uri]["repost_user_unobservable"] = False
                        reposters.update(reposted_by)

                    pbar.update(1)
                    pbar.set_postfix(
                        no_reposters=no_reposter_count,
                        refresh=False
                    )

                await asyncio.sleep(0)

    print(
        f"Reposts unobservable: {no_reposter_count}/{total} "
        f"({100 * no_reposter_count / total:.2f}%)"
    )

    return reposters


# --------------------------------------------------
# FETCH PROFILE / FOLLOWS / HISTORY
# --------------------------------------------------

async def fetch_profile(session, did):
    try:
        async with session.get(
            PROFILE_API, params={"actor": did}, headers=HEADERS
        ) as r:
            if r.status != 200:
                return None
            return await r.json()
    except (asyncio.TimeoutError, aiohttp.ClientError):
        return None


async def fetch_followed_authors(session, did, author_set):
    try:
        async with session.get(
            FOLLOW_API,
            params={"actor": did, "limit": 100},
            headers=HEADERS
        ) as r:
            if r.status != 200:
                return []
            data = await r.json()
            return [
                u["did"] for u in data.get("follows", [])
                if u.get("did") in author_set
            ]
    except (asyncio.TimeoutError, aiohttp.ClientError):
        return []


async def fetch_history(session, did, limit=50):
    history, cursor = [], None

    while len(history) < limit:
        params = {"actor": did, "limit": min(100, limit - len(history))}
        if cursor:
            params["cursor"] = cursor

        try:
            async with session.get(FEED_API, params=params, headers=HEADERS) as r:
                if r.status != 200:
                    break

                data = await r.json()
                feed = data.get("feed", [])
                if not feed:
                    break

                for item in feed:
                    post = item.get("post")
                    if not post:
                        continue

                    record = post.get("record", {})
                    reason = item.get("reason", {})

                    if reason.get("$type", "").endswith("reasonRepost"):
                        history.append({
                            "type": "repost",
                            "post_uri": post.get("uri"),
                            "post_author_did": post.get("author", {}).get("did"),
                            "reposted_at": reason.get("indexedAt"),
                        })
                        if len(history) >= limit:
                            break
                        continue

                    if "reply" not in record:
                        history.append({
                            "type": "post",
                            "post_uri": post.get("uri"),
                            "created_at": record.get("createdAt"),
                            "text": record.get("text", ""),
                            "likes": post.get("likeCount"),
                            "reposts": post.get("repostCount"),
                            "replies": post.get("replyCount"),
                            "quotes": post.get("quoteCount"),
                        })

                    if len(history) >= limit:
                        break

                cursor = data.get("cursor")
                if not cursor:
                    break

        except (asyncio.TimeoutError, aiohttp.ClientError):
            break

        await asyncio.sleep(0.01)

    return history


# --------------------------------------------------
# MAIN BUILDER
# --------------------------------------------------

async def build_users_from_posts(posts_dict, concurrency=200):
    author_set = get_author_dids(posts_dict)
    reposters = await collect_all_reposters(posts_dict)

    user_dids = list(reposters | author_set)
    users = {}

    connector = aiohttp.TCPConnector(
        limit=300, limit_per_host=80, ttl_dns_cache=300
    )
    timeout = aiohttp.ClientTimeout(total=60, connect=10, sock_read=30)

    follows_limiter = aiolimiter.AsyncLimiter(90, 1)
    sem = asyncio.Semaphore(concurrency)

    post_uri_set = set(posts_dict.keys())

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout
    ) as session:

        async def limited_follows(did):
            async with follows_limiter:
                return await fetch_followed_authors(
                    session, did, author_set
                )

        async def process(did):
            try:
                # only protect task creation
                async with sem:
                    profile_t = asyncio.create_task(
                        fetch_profile(session, did)
                    )
                    history_t = asyncio.create_task(
                        fetch_history(session, did, limit=50)
                    )
                    follows_t = asyncio.create_task(
                        limited_follows(did)
                    )

                profile, history, follows = await asyncio.gather(
                    profile_t, history_t, follows_t,
                    return_exceptions=True
                )

                if not profile or isinstance(profile, Exception):
                    return

                if isinstance(history, Exception):
                    history = []

                if isinstance(follows, Exception):
                    follows = []

            except Exception:
                return

            reposts_of_posts = [
                {
                    "post_uri": h.get("post_uri"),
                    "reposted_at": h.get("reposted_at"),
                }
                for h in history
                if h.get("type") == "repost"
                and h.get("post_uri") in post_uri_set
            ]

            created = profile.get("createdAt")
            age_days = (
                max(
                    1,
                    (datetime.now(timezone.utc) - parse_dt(created)).days
                )
                if created else None
            )

            users[did] = {
                "profile": {
                    "did": did,
                    "handle": profile.get("handle"),
                    "display_name": profile.get("displayName"),
                    "description": profile.get("description"),
                    "created_at": created,
                },
                "stats": {
                    "followers": profile.get("followersCount"),
                    "follows": profile.get("followsCount"),
                    "posts": profile.get("postsCount"),
                    "account_age_days": age_days,
                },
                "follows_authors": follows,
                "reposts_of_posts": reposts_of_posts,
                "history": history,
            }

        CHUNK_SIZE = 800
        total = len(user_dids)

        with tqdm(total=total, desc="Building users", unit="user") as pbar:
            for i in range(0, total, CHUNK_SIZE):
                chunk = user_dids[i:i + CHUNK_SIZE]
                pending = {asyncio.create_task(process(d)) for d in chunk}

                while pending:
                    done, pending = await asyncio.wait(
                        pending,
                        return_when=asyncio.FIRST_COMPLETED
                    )
                    pbar.update(len(done))

                await asyncio.sleep(0)

    return users


In [71]:
# =====================================================
# BLUESKY DATASET BUILDER (posts_dict -> users)
# FINAL VERSION (timezone-safe, progress bars, stable)
# =====================================================

import aiohttp
import asyncio
from datetime import datetime, timezone
from tqdm import tqdm
import aiolimiter

REPOST_API  = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"
PROFILE_API = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
FEED_API    = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
FOLLOW_API  = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

HEADERS = {"User-Agent": "repost-prediction-research/1.0"}

# -------------------------
# UTILS (TIMEZONE SAFE)
# -------------------------

def parse_dt(ts: str):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)

def get_author_dids(posts_dict):
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors

# -------------------------
# FETCH REPOSTERS
# -------------------------

async def fetch_reposters(session, uri):
    params = {"uri": uri, "limit": 100}
    try:
        async with session.get(REPOST_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception:
        return []

async def collect_reposter_dict(posts_dict, concurrency=25):
    posts, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:
        for uri, post in posts_dict.items():
            if post.get("repostCount", 0) > 0:
                posts.append(uri)
                tasks.append(fetch_reposters(session, uri))

        reposter_dict = {}
        failed_posts = 0

        pbar = tqdm(posts, desc="Fetching reposters", unit="post")

        for uri, task in zip(pbar, asyncio.as_completed(tasks)):
            reposters = await task

            if not reposters:
                failed_posts += 1
                pbar.set_postfix(
                    failed=f"{failed_posts}/{len(posts)}",
                    refresh=False
                )
                continue

            for r in reposters:
                reposter_dict.setdefault(r, []).append(uri)

    print(
        f"\nReposter fetch summary: "
        f"{failed_posts}/{len(posts)} posts "
        f"({failed_posts/len(posts):.1%}) returned no reposters"
    )

    return reposter_dict

# -------------------------
# FOLLOW FETCHING (UNCHANGED)
# -------------------------

async def fetch_follows(session, reposter, author_set, retries=3, first_page_only=True):
    follows = set()
    cursor = None
    headers = {"User-Agent": "Mozilla/5.0"}
    delay = 1

    for attempt in range(retries):
        try:
            while True:
                params = {"actor": reposter, "limit": 100}
                if cursor:
                    params["cursor"] = cursor

                async with session.get(FOLLOW_API, params=params, headers=headers) as r:
                    if r.status != 200:
                        if 500 <= r.status < 600 and attempt < retries - 1:
                            await asyncio.sleep(delay)
                            delay *= 2
                            continue
                        return reposter, []

                    data = await r.json()
                    follows.update(
                        u["did"] for u in data.get("follows", [])
                        if u.get("did") in author_set
                    )

                    cursor = data.get("cursor")
                    if not cursor or first_page_only:
                        break

            return reposter, list(follows)

        except Exception:
            if attempt < retries - 1:
                await asyncio.sleep(delay)
                delay *= 2
                continue
            return reposter, []

async def collect_reposter_followed_authors(
    reposter_dict,
    posts_dict,
    concurrency=100,
    reqs_per_sec=50
):
    reposters = list(reposter_dict.keys())
    total = len(reposters)
    author_set = get_author_dids(posts_dict)
    counter = {"done": 0}

    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)
    connector = aiohttp.TCPConnector(limit=300, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10, connect=5, sock_read=5)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem, rate_limiter:
                rep, authors = await fetch_follows(session, reposter, author_set)
                counter["done"] += 1
                if counter["done"] % 100 == 0 or counter["done"] == total:
                    print(
                        f"\rFetching followed authors: "
                        f"{counter['done']}/{total} "
                        f"({counter['done']/total:.1%})",
                        end="",
                        flush=True
                    )
                return rep, authors

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks)

    print()
    return {r: authors for r, authors in responses if authors}

# -------------------------
# FETCH PROFILE
# -------------------------

async def fetch_profile(session, did):
    async with session.get(PROFILE_API, params={"actor": did}, headers=HEADERS) as r:
        if r.status != 200:
            return None
        return await r.json()

# -------------------------
# FETCH USER HISTORY (≤50)
# -------------------------

async def fetch_user_history(session, did, limit=50):
    history = []
    cursor = None

    while len(history) < limit:
        params = {"actor": did, "limit": min(100, limit - len(history))}
        if cursor:
            params["cursor"] = cursor

        async with session.get(FEED_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                break

            data = await r.json()
            feed = data.get("feed", [])

            for item in feed:
                post = item.get("post")
                if not post:
                    continue

                record = post.get("record", {})
                reason = item.get("reason", {})

                if reason.get("$type", "").endswith("reasonRepost"):
                    activity_type = "repost"
                    parent_post_uri = post["uri"]
                    parent_author_did = post["author"]["did"]
                    reposted_at = reason.get("indexedAt")

                elif "reply" not in record:
                    activity_type = "post"
                    parent_post_uri = None
                    parent_author_did = None
                    reposted_at = None
                else:
                    continue

                facets = record.get("facets", [])
                has_links = any(
                    f["features"][0]["$type"].endswith("link")
                    for f in facets
                )

                embed = record.get("embed", {})
                media_type = None
                media_count = 0

                if embed:
                    et = embed.get("$type", "")
                    if et.endswith("images"):
                        media_type = "image"
                        media_count = len(embed.get("images", []))
                    elif et.endswith("video"):
                        media_type = "video"
                        media_count = 1
                    elif et.endswith("external"):
                        media_type = "external"
                        media_count = 1
                    elif et.endswith("recordWithMedia"):
                        media_type = "mixed"
                        media_count = 1

                history.append({
                    "activity_type": activity_type,
                    "created_at": record.get("createdAt"),
                    "reposted_at": reposted_at,
                    "post_uri": post["uri"],
                    "post_author_did": post["author"]["did"],
                    "parent_post_uri": parent_post_uri,
                    "parent_author_did": parent_author_did,
                    "text": record.get("text", ""),
                    "langs": record.get("langs", []),
                    "like_count": post.get("likeCount"),
                    "repost_count": post.get("repostCount"),
                    "reply_count": post.get("replyCount"),
                    "quote_count": post.get("quoteCount"),
                    "has_links": has_links,
                    "media_type": media_type,
                    "media_count": media_count,
                    "labels": post.get("labels", []),
                })

                if len(history) >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                break

    return history

# -------------------------
# MAIN BUILDER
# -------------------------

async def build_users_from_posts(posts_dict):
    # -------------------------
    # STAGE 1 + 2 (UNCHANGED)
    # -------------------------
    reposter_dict = await collect_reposter_dict(posts_dict)
    followed_authors = await collect_reposter_followed_authors(
        reposter_dict, posts_dict
    )

    # -------------------------
    # BUILD USER SET
    # -------------------------
    user_dids = set(reposter_dict.keys())
    user_dids |= get_author_dids(posts_dict)

    users = {}

    # -------------------------
    # STAGE 3 — SAFE USER BUILD
    # -------------------------
    connector = aiohttp.TCPConnector(
        limit=200,
        limit_per_host=50,
        ttl_dns_cache=300
    )

    timeout = aiohttp.ClientTimeout(
        total=15,
        connect=5,
        sock_read=5
    )

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout
    ) as session:

        sem = asyncio.Semaphore(100)  # HARD CONCURRENCY CAP

        async def process_user(did):
            async with sem:
                try:
                    profile = await fetch_profile(session, did)
                    if not profile:
                        return

                    history = await fetch_user_history(session, did)
                    created = profile.get("createdAt")

                    age_days = (
                        max(1, (datetime.now(timezone.utc) - parse_dt(created)).days)
                        if created else None
                    )

                    users[did] = {
                        "profile": {
                            "did": did,
                            "handle": profile.get("handle"),
                            "display_name": profile.get("displayName"),
                            "description": profile.get("description"),
                            "created_at": created,
                        },
                        "stats": {
                            "followers": profile.get("followersCount"),
                            "follows": profile.get("followsCount"),
                            "posts": profile.get("postsCount"),
                            "account_age_days": age_days,
                        },
                        "history": history,
                        "reposted_posts": reposter_dict.get(did, []),
                        "follows_authors": followed_authors.get(did, []),
                    }

                except asyncio.TimeoutError:
                    return
                except Exception:
                    return

        tasks = [process_user(d) for d in user_dids]

        for task in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Building users",
            unit="user"
        ):
            await task

    return users


In [6]:
# =====================================================
# BLUESKY DATASET BUILDER (posts_dict -> users)
# FINAL VERSION (timezone-safe, progress bars, stable)
# =====================================================

import aiohttp
import asyncio
from datetime import datetime, timezone
from tqdm import tqdm
import aiolimiter

REPOST_API  = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"
PROFILE_API = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
FEED_API    = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
FOLLOW_API  = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

HEADERS = {"User-Agent": "repost-prediction-research/1.0"}

# -------------------------
# UTILS (TIMEZONE SAFE)
# -------------------------

def parse_dt(ts: str):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)

def get_author_dids(posts_dict):
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors


def flatten_posts_dict(posts_by_tag):
    flat_posts = {}
    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            flat_posts[uri] = post
    return flat_posts
# -------------------------
# FETCH REPOSTERS
# -------------------------

async def fetch_reposters(session, uri):
    try:
        async with session.get(
            REPOST_API,
            params={"uri": uri, "limit": 100},
            headers=HEADERS
        ) as r:
            if r.status != 200:
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception:
        return []

async def collect_reposter_dict(posts_dict, concurrency=25):
    posts, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:
        for uri, post in posts_dict.items():
            if post.get("repostCount", 0) > 0:
                posts.append(uri)
                tasks.append(fetch_reposters(session, uri))

        reposter_dict = {}
        failed_posts = 0

        pbar = tqdm(posts, desc="Fetching reposters", unit="post")

        for uri, task in zip(pbar, asyncio.as_completed(tasks)):
            reposters = await task

            # 🔹 CHANGE 1: store reposter DIDs on the post
            posts_dict[uri]["reposted_by"] = reposters

            if not reposters:
                posts_dict[uri]["repost_user_unobservable"] = True
                failed_posts += 1
                pbar.set_postfix(
                    failed=f"{failed_posts}/{len(posts)}",
                    refresh=False
                )
                continue

            posts_dict[uri]["repost_user_unobservable"] = False

            for r in reposters:
                reposter_dict.setdefault(r, []).append(uri)

    print(
        f"\nReposter fetch summary: "
        f"{failed_posts}/{len(posts)} posts "
        f"({failed_posts/len(posts):.1%}) returned no reposters"
    )

    return reposter_dict

# -------------------------
# FOLLOW FETCHING
# -------------------------

async def fetch_follows(session, did, author_set, retries=3, first_page_only=True):
    follows = set()
    cursor = None
    delay = 1

    for attempt in range(retries):
        try:
            while True:
                params = {"actor": did, "limit": 100}
                if cursor:
                    params["cursor"] = cursor

                async with session.get(FOLLOW_API, params=params, headers=HEADERS) as r:
                    if r.status != 200:
                        if 500 <= r.status < 600 and attempt < retries - 1:
                            await asyncio.sleep(delay)
                            delay *= 2
                            continue
                        return did, []

                    data = await r.json()
                    follows.update(
                        u["did"] for u in data.get("follows", [])
                        if u.get("did") in author_set
                    )

                    cursor = data.get("cursor")
                    if not cursor or first_page_only:
                        break

            return did, list(follows)

        except Exception:
            if attempt < retries - 1:
                await asyncio.sleep(delay)
                delay *= 2
                continue
            return did, []

async def collect_reposter_followed_authors(
    reposter_dict,
    posts_dict,
    concurrency=100,
    reqs_per_sec=50
):
    # 🔹 CHANGE 2: authors ∪ reposters
    user_dids = set(reposter_dict.keys())
    user_dids |= get_author_dids(posts_dict)
    user_dids = list(user_dids)

    total = len(user_dids)
    author_set = get_author_dids(posts_dict)
    counter = {"done": 0}

    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)
    connector = aiohttp.TCPConnector(limit=300, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10, connect=5, sock_read=5)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(did):
            async with sem, rate_limiter:
                rep, authors = await fetch_follows(session, did, author_set)
                counter["done"] += 1
                if counter["done"] % 100 == 0 or counter["done"] == total:
                    print(
                        f"\rFetching followed authors: "
                        f"{counter['done']}/{total} "
                        f"({counter['done']/total:.1%})",
                        end="",
                        flush=True
                    )
                return rep, authors

        tasks = [limited_fetch(d) for d in user_dids]
        responses = await asyncio.gather(*tasks)

    print()
    return {r: authors for r, authors in responses if authors}

# -------------------------
# FETCH PROFILE
# -------------------------

async def fetch_profile(session, did):
    async with session.get(PROFILE_API, params={"actor": did}, headers=HEADERS) as r:
        if r.status != 200:
            return None
        return await r.json()

# -------------------------
# FETCH USER HISTORY (UNCHANGED)
# -------------------------

async def fetch_user_history(session, did, limit=50):
    history = []
    cursor = None

    while len(history) < limit:
        params = {"actor": did, "limit": min(100, limit - len(history))}
        if cursor:
            params["cursor"] = cursor

        async with session.get(FEED_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                break

            data = await r.json()
            feed = data.get("feed", [])

            for item in feed:
                post = item.get("post")
                if not post:
                    continue

                record = post.get("record", {})
                reason = item.get("reason", {})

                if reason.get("$type", "").endswith("reasonRepost"):
                    activity_type = "repost"
                    parent_post_uri = post["uri"]
                    parent_author_did = post["author"]["did"]
                    reposted_at = reason.get("indexedAt")

                elif "reply" not in record:
                    activity_type = "post"
                    parent_post_uri = None
                    parent_author_did = None
                    reposted_at = None
                else:
                    continue

                facets = record.get("facets", [])
                has_links = any(
                    f["features"][0]["$type"].endswith("link")
                    for f in facets
                )

                embed = record.get("embed", {})
                media_type = None
                media_count = 0

                if embed:
                    et = embed.get("$type", "")
                    if et.endswith("images"):
                        media_type = "image"
                        media_count = len(embed.get("images", []))
                    elif et.endswith("video"):
                        media_type = "video"
                        media_count = 1
                    elif et.endswith("external"):
                        media_type = "external"
                        media_count = 1
                    elif et.endswith("recordWithMedia"):
                        media_type = "mixed"
                        media_count = 1

                history.append({
                    "activity_type": activity_type,
                    "created_at": record.get("createdAt"),
                    "reposted_at": reposted_at,
                    "post_uri": post["uri"],
                    "post_author_did": post["author"]["did"],
                    "parent_post_uri": parent_post_uri,
                    "parent_author_did": parent_author_did,
                    "text": record.get("text", ""),
                    "langs": record.get("langs", []),
                    "like_count": post.get("likeCount"),
                    "repost_count": post.get("repostCount"),
                    "reply_count": post.get("replyCount"),
                    "quote_count": post.get("quoteCount"),
                    "has_links": has_links,
                    "media_type": media_type,
                    "media_count": media_count,
                    "labels": post.get("labels", []),
                })

                if len(history) >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                break

    return history

# -------------------------
# MAIN BUILDER
# -------------------------

async def build_users_from_posts(posts_dict):
    reposter_dict = await collect_reposter_dict(posts_dict)
    followed_authors = await collect_reposter_followed_authors(
        reposter_dict, posts_dict
    )

    user_dids = set(reposter_dict.keys())
    user_dids |= get_author_dids(posts_dict)

    users = {}

    connector = aiohttp.TCPConnector(limit=200, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=15, connect=5, sock_read=5)

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout
    ) as session:

        sem = asyncio.Semaphore(100)

        async def process_user(did):
            async with sem:
                try:
                    profile = await fetch_profile(session, did)
                    if not profile:
                        return

                    history = await fetch_user_history(session, did)
                    created = profile.get("createdAt")

                    age_days = (
                        max(1, (datetime.now(timezone.utc) - parse_dt(created)).days)
                        if created else None
                    )

                    users[did] = {
                        "profile": {
                            "did": did,
                            "handle": profile.get("handle"),
                            "display_name": profile.get("displayName"),
                            "description": profile.get("description"),
                            "created_at": created,
                        },
                        "stats": {
                            "followers": profile.get("followersCount"),
                            "follows": profile.get("followsCount"),
                            "posts": profile.get("postsCount"),
                            "account_age_days": age_days,
                        },
                        "history": history,
                        "reposted_posts": reposter_dict.get(did, []),
                        "follows_authors": followed_authors.get(did, []),
                    }

                except Exception:
                    return

        tasks = [process_user(d) for d in user_dids]

        for task in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Building users",
            unit="user"
        ):
            await task

    return users


In [7]:
post_dict = flatten_posts_dict(posts)
users = await build_users_from_posts(post_dict)

Fetching reposters: 100%|██████████| 42625/42625 [05:01<00:00, 141.20post/s, failed=6142/42625] 



Reposter fetch summary: 6142/42625 posts (14.4%) returned no reposters
Fetching followed authors: 74868/74868 (100.0%)


Building users: 100%|██████████| 74868/74868 [43:11<00:00, 28.88user/s]    


In [8]:
with open("TEST_X2.json", "w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)


In [9]:
def print_basic_stats(users, posts_dict):
    # -------------------------
    # Authors followed per user
    # -------------------------
    follow_counts = [
        len(u.get("follows_authors", []))
        for u in users.values()
    ]

    avg_authors_followed = (
        sum(follow_counts) / len(follow_counts)
        if follow_counts else 0
    )
    max_authors_followed = max(follow_counts) if follow_counts else 0
    # -------------------------
    # Reposts per post
    # -------------------------
    repost_counts = [
        len(p.get("reposted_by", []))
        for p in posts_dict.values()
        if "reposted_by" in p
    ]

    avg_reposts_per_post = (
        sum(repost_counts) / len(repost_counts)
        if repost_counts else 0
    )
    max_reposts_per_post = max(repost_counts) if repost_counts else 0

    posts_with_reposts = sum(c > 0 for c in repost_counts)
    pct_posts_with_reposts = (
        100 * posts_with_reposts / len(post_dict)
        if repost_counts else 0
    )

    # -------------------------
    # Reposts per user (from reposter_dict output)
    # -------------------------
    reposts_per_user = [
        len(u.get("reposted_posts", []))
        for u in users.values()
    ]

    users_with_reposts = sum(c > 0 for c in reposts_per_user)
    pct_users_with_reposts = (
        100 * users_with_reposts / len(reposts_per_user)
        if reposts_per_user else 0
    )

    # -------------------------
    # Repost timestamps (from history)
    # -------------------------
    total_reposts = 0
    reposts_with_time = 0

    for u in users.values():
        for h in u.get("history", []):
            if h.get("activity_type") == "repost":
                total_reposts += 1
                if h.get("reposted_at"):
                    reposts_with_time += 1

    pct_reposts_with_time = (
        100 * reposts_with_time / total_reposts
        if total_reposts else 0
    )

    # -------------------------
    # Print results
    # -------------------------
    print("===== BASIC DATASET STATS =====")
    print(f"Users: {len(users)}")
    print(f"Posts: {len(posts_dict)}")
    print()

    print(
        f"Authors followed per user → "
        f"avg: {avg_authors_followed:.2f}, "
        f"max: {max_authors_followed}"
    )

    print(
        f"Reposts per post → "
        f"avg: {avg_reposts_per_post:.2f}, "
        f"max: {max_reposts_per_post}"
    )

    print(
        f"Posts with ≥1 repost → "
        f"{posts_with_reposts} "
        f"({pct_posts_with_reposts:.2f}%)"
    )

    print(
        f"Users with ≥1 repost → "
        f"{users_with_reposts} "
        f"({pct_users_with_reposts:.2f}%)"
    )

    print(
        f"Reposts with timestamp → "
        f"{reposts_with_time}/{total_reposts} "
        f"({pct_reposts_with_time:.2f}%)"
    )


In [10]:
def print_posts_with_more_than_x_reposts(posts_dict,x):
    count = sum(
        p.get("repostCount", 0) > x
        for p in posts_dict.values()
    )
    print(f"Posts with >{x} reposts: {count}/{len(posts_dict)}")


print_posts_with_more_than_x_reposts(post_dict,0)

Posts with >0 reposts: 42625/169904


In [11]:
print_basic_stats(users,post_dict)

===== BASIC DATASET STATS =====
Users: 74383
Posts: 169904

Authors followed per user → avg: 5.24, max: 68
Reposts per post → avg: 3.51, max: 100
Posts with ≥1 repost → 36483 (21.47%)
Users with ≥1 repost → 50850 (68.36%)
Reposts with timestamp → 2285844/2285844 (100.00%)


In [12]:
def count_unique_authors(posts_dict):
    """
    Returns the number of unique author DIDs in posts_dict.
    """
    return len({
        (
            p.get("author", {}).get("did")
            or p.get("post", {}).get("author", {}).get("did")
        )
        for p in posts_dict.values()
        if p
    })
num_authors = count_unique_authors(post_dict)
print(f"Unique authors: {num_authors}")


Unique authors: 29746
